# Used Standard Scaler in this notebook
### Tested the following parameters combinations
- eps = 1.725
    - min_samples = 1, thresholds = 0.115, 0.1175, 0.12
- eps = 1.73
    - min_samples = 1, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.1175, 0.12, 0.125
    - min_samples = 2, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
    - min_samples = 3, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
- eps = 1.735
    - min_samples = 1, thresholds = 0.115, 0.1175, 0.12, 0.125, 0.13, 0.135
- eps = 1.74
    - min_samples = 1, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13, 0.135, 0.14
    - min_samples = 2, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
    - min_samples = 3, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
- eps = 1.745
    - min_samples = 1, thresholds = 0.125, 0.13, 0.135, 0.14
- eps = 1.75
    - min_samples = 1, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13, 0.135, 0.14
    - min_samples = 2, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
    - min_samples = 3, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
- eps = 1.76
    - min_samples = 1, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13
    - min_samples = 2, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
    - min_samples = 3, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
- eps = 1.77
    - min_samples = 1, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13
    - min_samples = 2, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
    - min_samples = 3, thresholds = 0.09, 0.095, 0.1, 0.105, 0.11
- eps = 1.78
    - min_samples = 1, thresholds = 0.105, 0.11, 0.115, 0.12, 0.125, 0.13

### Result of these evaluation is stored in `standard-scaler-results.txt` file

### Best params found
`Params: eps=1.74, min_samples=1, threshold=0.12`
```
F1: 0.6407, Recall: 0.8102, Precision: 0.5298, Accuracy: 0.9890
confusion matrix:

Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  436313    3883
Fraud        1025    4375
```

In [ ]:
import cudf
import cuml 
import cupy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.model_selection import train_test_split
from cuml.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from cuml.decomposition import PCA
from cuml.cluster import DBSCAN
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import ParameterGrid
import numpy as np
import warnings
import kagglehub
warnings.filterwarnings('ignore')
from joblib import Parallel, delayed

In [ ]:
cuml.set_global_output_type('cupy')

In [ ]:
def draw_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Fraud', 'Fraud'], yticklabels=['Not Fraud', 'Fraud'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
def print_confusion_matrix(cm, title):
    print(f"{title}:")
    print("         Predicted")
    print("         Not Fraud  Fraud")
    print("Actual")
    print(f"Not Fraud  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"Fraud      {cm[1,0]:6d}  {cm[1,1]:6d}")

In [ ]:
path = kagglehub.dataset_download("ealaxi/banksim1")
print("Path to dataset files:", path)

In [ ]:
data = cudf.read_csv(path + '/bs140513_032310.csv')

In [ ]:
data = data.drop(['zipcodeOri', 'zipMerchant'], axis=1)
data['customer'] = data['customer'].str[1:-1]
data['age'] = data['age'].str[1:-1]
data['gender'] = data['gender'].str[1:-1]
data['merchant'] = data['merchant'].str[1:-1]
data['category'] = data['category'].str.replace('es_', '')
data = data[data['gender'] != 'U']
data['age'] = data['age'].replace('U', '7')
data['age'] = data['age'].astype('float32')

In [ ]:
data['amount'] = data['amount'].astype('float32')
data['amount_thresh'] = cudf.Series('>3000', index=data.index)
data.loc[data['amount'] <= 500, 'amount_thresh'] = '0-500'
data.loc[(data['amount'] > 500) & (data['amount'] <= 1000), 'amount_thresh'] = '500-1000'
data.loc[(data['amount'] > 1000) & (data['amount'] <= 1500), 'amount_thresh'] = '1000-1500'
data.loc[(data['amount'] > 1500) & (data['amount'] <= 2000), 'amount_thresh'] = '1500-2000'
data.loc[(data['amount'] > 2000) & (data['amount'] <= 2500), 'amount_thresh'] = '2000-2500'
data.loc[(data['amount'] > 2500) & (data['amount'] <= 3000), 'amount_thresh'] = '2500-3000'

In [ ]:
customer = data.groupby('customer').size().reset_index(name='customer_total_trans')
customer['customer_ID'] = cp.arange(1, len(customer) + 1)

merchant = data.groupby('merchant').size().reset_index(name='merchant_total_trans')
merchant['merchant_ID'] = cp.arange(1, len(merchant) + 1)

category = data.groupby('category').size().reset_index(name='category_total_trans')
category['category_ID'] = cp.arange(1, len(category) + 1)

amount_thresh = data.groupby('amount_thresh').size().reset_index(name='amount_thresh_total_trans')
amount_thresh['amount_thresh_ID'] = cp.arange(1, len(amount_thresh) + 1)

In [ ]:
data = data.merge(customer, on='customer').merge(merchant, on='merchant')
data = data.drop(['customer', 'merchant'], axis=1)
data = data.merge(category, on='category').merge(amount_thresh, on='amount_thresh')
data['category'] = data['category_ID']
data['amount_thresh'] = data['amount_thresh_ID']
data = data.drop(['category_ID', 'amount_thresh_ID'], axis=1)

In [ ]:
age = data.groupby('age').size().reset_index(name='age_total_trans')
gender = data.groupby('gender').size().reset_index(name='gender_total_trans')
data = data.merge(age, on='age').merge(gender, on='gender')

In [ ]:
total_freq = len(data)
data['merchant_total_trans'] = (data['merchant_total_trans'] / total_freq * 100).round(5)
data['category_total_trans'] = (data['category_total_trans'] / total_freq * 100).round(5)
data['amount_thresh_total_trans'] = (data['amount_thresh_total_trans'] / total_freq * 100).round(5)
data['age_total_trans'] = (data['age_total_trans'] / total_freq * 100).round(5)
data['gender_total_trans'] = (data['gender_total_trans'] / total_freq * 100).round(5)

In [ ]:
data = data.drop('step', axis=1)
data = data[['fraud'] + [col for col in data.columns if col != 'fraud']]

In [ ]:
ohe = OneHotEncoder(sparse=True, handle_unknown='ignore')
ohe_cols = ['age', 'gender', 'category', 'amount_thresh']
ohe_data = ohe.fit_transform(data[ohe_cols])
ohe_feature_names = ohe.get_feature_names(ohe_cols)
ohe_data = cudf.DataFrame(ohe_data.toarray(), columns=ohe_feature_names)
data = cudf.concat([data.drop(ohe_cols, axis=1), ohe_data], axis=1)

In [ ]:
le = LabelEncoder()
data['fraud'] = le.fit_transform(data['fraud'])

print("Data dimensions:", data.shape)

In [ ]:
pca_cols = ['age_0.0', 'age_1.0', 'age_2.0', 'age_3.0', 'age_4.0', 'age_5.0', 'age_6.0', 'age_7.0',
            'gender_E', 'gender_F', 'gender_M',
            'category_1', 'category_2', 'category_3', 'category_4', 'category_5', 'category_6', 'category_7',
            'category_8', 'category_9', 'category_10', 'category_11', 'category_12', 'category_13', 'category_14',
            'amount', 'amount_thresh_1', 'amount_thresh_2', 'amount_thresh_3', 'amount_thresh_4', 'amount_thresh_5',
            'amount_thresh_6', 'amount_thresh_7']
scaler = StandardScaler()
scaled_data_big = scaler.fit_transform(data[pca_cols])
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data_big)
pca_df = cudf.DataFrame(pca_data, columns=['PC1', 'PC2'])
pca_df['fraud'] = data['fraud']

In [ ]:
pca_df_np = pca_df.to_pandas()
plt.figure(figsize=(10, 6))
sns.scatterplot(x='PC1', y='PC2', hue='fraud', style='fraud', data=pca_df_np)
plt.title('Fraud Spread over the First 2 Principal Components')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()
print("Explained variance ratio:", cp.asnumpy(pca.explained_variance_ratio_))

In [ ]:
X = data.drop('fraud', axis=1)
y = data['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123, stratify=y)

In [ ]:
X_sample = X_train.copy()
y_sample = y_train.copy()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

In [ ]:
def evaluate_params(params):
    dbscan = DBSCAN(eps=params['eps'], min_samples=params['min_samples'])
    labels = dbscan.fit_predict(X_scaled, out_dtype='int64')

    if len(cp.unique(labels)) <= 1:
        return None

    temp_data = cudf.DataFrame({
        'cluster': labels,
        'true_label': y_sample
    })

    cluster_mapping = {}
    for cluster in cp.unique(labels).get():
        if cluster != -1:
            mask = temp_data['cluster'] == cluster
            fraud_ratio = temp_data[mask]['true_label'].mean()
            cluster_mapping[cluster] = 1 if fraud_ratio > params['fraud_threshold'] else 0

    temp_data['predicted'] = temp_data['cluster'].map(cluster_mapping).fillna(0)
    y_true_cp = cp.array(y_sample.astype('int32'))
    y_pred_cp = cp.array(temp_data['predicted'].astype('int32'))

    y_true_np = y_true_cp.get()
    y_pred_np = y_pred_cp.get()

    f1 = float(f1_score(y_true_np, y_pred_np))
    recall = float(recall_score(y_true_np, y_pred_np))
    precision = float(precision_score(y_true_np, y_pred_np))
    accuracy = float(accuracy_score(y_true_np, y_pred_np))

    print(f"Params: eps={params['eps']}, min_samples={params['min_samples']}, "
          f"threshold={params['fraud_threshold']}")
    print(f"  F1: {f1:.4f}, Recall: {recall:.4f}, "
          f"Precision: {precision:.4f}, Accuracy: {accuracy:.4f}")
    print("confusion matrix:\n")
    cm = confusion_matrix(y_true_np, y_pred_np)
    print_confusion_matrix(cm, "Confusion Matrix")

    print()

    return {
        'params': params,
        'f1': f1,
        'recall': recall,
        'precision': precision,
        'accuracy': accuracy
    }

In [ ]:
def print_all_model_results(results):
  for res in results:
      print(f"Params: eps={res['params']['eps']}, min_samples={res['params']['min_samples']}, "
            f"threshold={res['params']['fraud_threshold']}")
      print(f"  F1: {res['f1']:.4f}, Recall: {res['recall']:.4f}, "
            f"Precision: {res['precision']:.4f}, Accuracy: {res['accuracy']:.4f}")
      print()

In [ ]:
def print_best_model(results):
  best_result = max(results, key=lambda r: (r['f1'], r['recall']))
  print("=== Best Model ===")
  print(f"Params: eps={best_result['params']['eps']}, min_samples={best_result['params']['min_samples']}, "
        f"threshold={best_result['params']['fraud_threshold']}")
  print(f"F1: {best_result['f1']:.4f}, Recall: {best_result['recall']:.4f}, "
        f"Precision: {best_result['precision']:.4f}, Accuracy: {best_result['accuracy']:.4f}")

In [ ]:
param_grid = {
    'eps': [1.725, 1.73, 1.735],  # Ultra-fine tuning
    'min_samples': [1],
    'fraud_threshold': [0.115, 0.1175, 0.12]  # Smaller threshold steps
}

In [ ]:
best_f1 = -1
best_recall = -1
best_params = None
best_labels = None
results = []
for params in ParameterGrid(param_grid):
    result = evaluate_params(params)
    if result is not None:
        results.append(result)

In [ ]:
print_best_model(results)

In [ ]:
print_all_model_results(results)

### Building the final model with the best found params . the best params is as follows
`` params = {
        'eps': 1.725,
        'min_samples': 1,
        'fraud_threshold': 0.115
    } ``

In [ ]:
def build_final_model(X_train, y_train, X_test, y_test):
    params = {
        'eps': 1.725,
        'min_samples': 1,
        'fraud_threshold': 0.115
    }
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    
    dbscan = DBSCAN(eps=params['eps'], min_samples=params['min_samples'])
    labels = dbscan.fit_predict(X_scaled, out_dtype='int64')
    
    temp_data = cudf.DataFrame({
        'cluster': labels,
        'true_label': y_train
    })
    
    cluster_mapping = {}
    for cluster in cp.unique(labels).get():
        if cluster != -1:
            mask = temp_data['cluster'] == cluster
            fraud_ratio = temp_data[mask]['true_label'].mean()
            cluster_mapping[cluster] = 1 if fraud_ratio > params['fraud_threshold'] else 0
    
    temp_data['predicted'] = temp_data['cluster'].map(cluster_mapping).fillna(0)
    
    y_train_np = y_train.to_array() if hasattr(y_train, 'to_array') else y_train.values.get()
    y_train_pred_np = temp_data['predicted'].to_array() if hasattr(temp_data['predicted'], 'to_array') else temp_data['predicted'].values.get()
    
    train_cm = confusion_matrix(y_train_np, y_train_pred_np)
    
    print("=== Training Results ===")
    print_confusion_matrix(train_cm, "Training Confusion Matrix")
    print(f"Recall: {recall_score(y_train_np, y_train_pred_np):.4f}")
    print(f"Precision: {precision_score(y_train_np, y_train_pred_np):.4f}")
    print(f"F1: {f1_score(y_train_np, y_train_pred_np):.4f}")
    
    X_test_scaled = scaler.transform(X_test)
    test_labels = dbscan.fit_predict(X_test_scaled, out_dtype='int64')
    
    test_data = cudf.DataFrame({
        'cluster': test_labels,
        'true_label': y_test
    })
    test_data['predicted'] = test_data['cluster'].map(cluster_mapping).fillna(0)
    
    y_test_np = y_test.to_array() if hasattr(y_test, 'to_array') else y_test.values.get()
    y_test_pred_np = test_data['predicted'].to_array() if hasattr(test_data['predicted'], 'to_array') else test_data['predicted'].values.get()
    
    test_cm = confusion_matrix(y_test_np, y_test_pred_np)
    
    print("\n=== Test Results ===")
    print_confusion_matrix(test_cm, "Test Confusion Matrix")
    print(f"Recall: {recall_score(y_test_np, y_test_pred_np):.4f}")
    print(f"Precision: {precision_score(y_test_np, y_test_pred_np):.4f}")
    print(f"F1: {f1_score(y_test_np, y_test_pred_np):.4f}")
    
    pca = PCA(n_components=2)
    pca_data = pca.fit_transform(X_scaled)
    pca_df = cudf.DataFrame(pca_data, columns=['PC1', 'PC2'])
    pca_df['cluster'] = labels
    pca_df['fraud'] = y_train
    
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='PC1', y='PC2', hue='cluster', style='fraud', 
                    data=pca_df.to_pandas(), palette='viridis')
    plt.title('Cluster Visualization with Fraud Labels (Train Data)')
    plt.show()
    
    return {
        'model': dbscan,
        'scaler': scaler,
        'cluster_mapping': cluster_mapping,
        'params': params
    }

final_model = build_final_model(X_train, y_train, X_test, y_test)

### But while testing the model on test data , the F1 score reduced significantly.
```
=== Training Results ===
Training Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  435632    4564
Fraud         948    4452
Recall: 0.8244
Precision: 0.4938
F1: 0.6176

=== Test Results ===
Test Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  139944    6788
Fraud         777    1023
Recall: 0.5683
Precision: 0.1310
F1: 0.2129
```

optimizing the function to get better results
- choosen min_samples as 2 for more robust clusters
- added cluster size and purity thresholds
- added AUC score for monitoring performance 

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
def build_optimized_model(X_train, y_train, X_test, y_test):
    params = {
        'eps': 1.72,  
        'min_samples': 2,  
        'fraud_threshold': 0.12  
    }
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    dbscan = DBSCAN(
        eps=params['eps'],
        min_samples=params['min_samples'],
        output_type='cupy'
    )
    
    train_labels = dbscan.fit_predict(X_train_scaled)
    test_labels = dbscan.fit_predict(X_test_scaled)
    
    def analyze_clusters(labels, true_labels):
        cluster_df = cudf.DataFrame({
            'cluster': labels,
            'true_label': true_labels
        })
        
        stats = []
        for cluster in cp.unique(labels).get():
            mask = cluster_df['cluster'] == cluster
            cluster_size = len(cluster_df[mask])
            if cluster_size < 10:  
                continue
            fraud_ratio = cluster_df[mask]['true_label'].mean()
            stats.append({
                'cluster': cluster,
                'fraud_ratio': fraud_ratio,
                'size': cluster_size,
                'stability': min(fraud_ratio, 1-fraud_ratio)  
            })
        return cudf.DataFrame(stats).sort_values('fraud_ratio', ascending=False)
    
    train_stats = analyze_clusters(train_labels, y_train)
    stable_clusters = train_stats[
        (train_stats['size'] > 20) & 
        (train_stats['stability'] > 0.7)
    ].to_pandas()
    
    cluster_mapping = {
        row['cluster']: 1 if row['fraud_ratio'] > params['fraud_threshold'] else 0
        for _, row in stable_clusters.iterrows()
    }
    cluster_mapping[-1] = 0 

    def evaluate(y_true, labels, name):
        pred = cudf.Series(labels).map(cluster_mapping).fillna(0)
        cm = confusion_matrix(y_true.to_array(), pred.to_array())
        
        print(f"\n=== {name} Results ===")
        print_confusion_matrix(cm, f"{name} Confusion Matrix")
        print(f"Recall: {recall_score(y_true.to_array(), pred.to_array()):.4f}")
        print(f"Precision: {precision_score(y_true.to_array(), pred.to_array()):.4f}")
        print(f"F1: {f1_score(y_true.to_array(), pred.to_array()):.4f}")
        print(f"AUC: {roc_auc_score(y_true.to_array(), pred.to_array()):.4f}")
        
        return pred
    
    train_pred = evaluate(y_train, train_labels, "Training")
    test_pred = evaluate(y_test, test_labels, "Test")
    
    plot_clusters(X_train_scaled, train_labels, y_train, "Training Clusters")
    plot_clusters(X_test_scaled, test_labels, y_test, "Test Clusters")
    
    return {
        'model': dbscan,
        'scaler': scaler,
        'cluster_mapping': cluster_mapping,
        'params': params,
        'cluster_stats': stable_clusters,
        'predictions': {
            'train': train_pred,
            'test': test_pred
        }
    }

def plot_clusters(X, labels, true_labels, title):
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(X) if "Train" in title else pca.transform(X)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        x=reduced[:, 0], 
        y=reduced[:, 1],
        hue=labels.get(),
        style=true_labels.to_array(),
        palette='viridis',
        alpha=0.6
    )
    plt.title(f"{title} (Explained Variance: {pca.explained_variance_ratio_.sum():.2f})")
    plt.show()

final_model = build_optimized_model(X_train, y_train, X_test, y_test)